we need to verify that the pulse sequences are valid, so we run the simulations for

- free diffusion in 3D (starting from a point source, diffusing freely at D0. This should recover D0)
- restricted diffusion inside a sphere (with varying spacing, we can compare to 1D solution)
- restricted diffusion inside an ellipsoid (recover the ellipsoid shape)

as these are fast, we can do 10 realisations with large number of particles

we can also compare this with main_randomwalk_CONVERGENCE

In [ ]:
import numpy as np
import math

mcse_40 = Sequence.MCSE(bfactor, scanner, NT, Gmax=40e-3)
mcse_80 = Sequence.MCSE(bfactor, scanner, NT, Gmax=80e-3)

D0 = 1e-9
for seq in (mcse_40, mcse_80):
    NP = 1e4
    x = np.zeros((NP, 3))  # position
    p = np.zeros((NP, 3))  # phase
    while seq.next():
        dt = seq.get_dt
        gG = seq.get_gG
        p = p + x*gG*dt
        x = x + np.random.randn(np.shape(x))*math.sqrt(2*D0*dt)

    res_seq = seq.computeResults(p)
    res_rms_Dx = np.mean(x[:,1]**2)/(2*sum(seq.dt))
    res_rms_Dy = np.mean(x[:,2]**2)/(2*sum(seq.dt))
    res_rms_Dz = np.mean(x[:,3]**2)/(2*sum(seq.dt))

#TODO: also verify stationary -> simply integrate the sequence

In [ ]:
def stationary(seq, NP=100000):
    xyz = np.zeros((NP, 3))
    # sum over time

In [ ]:
def analytical_plates(spacing, D0, seq=None, N_terms=100):
    """ Compute the analytical solution for diffusion between two infinite parallel plates """
    import numpy as np

    ii = np.arange(1, N_terms+1)

    if not seq:  # None
        return D0
    if isinf(spacing):  # no restriction
        return D0

    K = seq['delta'] * spacing * seq['gamma'] * seq['G_peak']  # aux variable
    e_term = np.exp(-(ii**2)*(pi**2)*D0*DELTA/(spacing**2))
    K_term = (1-((-1)**ii)*np.cos(K))/((K**2)-(ii*pi)**2)**2
    terms = e_term * K_term
    attenuation = 2*(1-np.cos(K))/(K**2) + 4*(K**2)*np.sum(terms)
    D = -np.log(attenuation)/seq['bfactor']

    return ADC


NOTE:

the analytical solution only applies in the short pulsed gradient limit, i.e. delta tends to dirac delta.

this explains why the PGSE sequence gives incorrect values as spacing -> inf (levels off above D0) ???

therefore, this is all meaningless and we will not compare the 1D solution to the analytical (except for STEAM and PGSE as reference)

also: remove the analytical thing and all its children from the sequence and create a separate function instead


In [ ]:
# sequences bvalue = 225e6
pgse  = seq={'delta': 0.1, 'gamma': 44, 'G_peak': 40e-3}
mcse  = seq={'delta': 0.1, 'gamma': 44, 'G_peak': 40e-3}
steam = seq={'delta': 0.1, 'gamma': 44, 'G_peak': 40e-3}

# ...
spacing = np.linspace(0, 0.001, 1000)
ADC_pgse, ADC_mcse, ADC_steam = np.empty((size.spacing, 3))
for i in range(spacing.size):
    N = 100000
    ADC_pgse[i]  = analytical_plates(spacing[i], D0=3, N_terms=N, seq=pgse)
    ADC_mcse[i]  = analytical_plates(spacing[i], D0=3, N_terms=N, seq=mcse)
    ADC_steam[i] = analytical_plates(spacing[i], D0=3, N_terms=N, seq=steam)

# ...
plt.figure()
plt.plot(spacing, ADC_pgse)
plt.plot(spacing, ADC_mcse)
plt.plot(spacing, ADC_steam)

In [ ]:
from process_DTI import calculate_tensor

directions = np.array([[1, 1, 0], [1, -1, 0], [1, 0, 1], [1, 0, -1], [0, 1, 1], [0, 1, -1]])
ndirs = directions.shape[0]
signal_ratio = np.zeros(ndirs)
for idir in range(ndirs):
    # bval
    if pulse_sequence_type == 'PGSE':
        bvalue = (gG**2) * ((delta**2)*(DELTA-delta/3))
    elif pulse_sequence_type == 'STEAM':
        bvalue = (gG**2) * ((delta**2)*(DELTA-delta/3) + ramp**3/30 - delta*ramp**2/6)
    b_factor = norm(directions[i, :])**2 * bvalue
    # signal
    particle_phase = np.sum(directions[idir] * phase, 1)
    signal_exp = np.sum(np.exp(-1i*particle_phase))
    signal_ratio[idir] = abs(signal_exp/size(phase, 1))

tensor_sim = calculate_tensor(signal_ratio, directions, bvalue=225e6)

# compare ADC along Cartesian directions
Dxyz_sim = tensor_sim.tensor[np.diag_indices(3)]
Dxyz_rms = np.array([rms_x, rms_y, rms_z])**2/(2*DELTA)


In [ ]:
import numpy as np

def appendSpherical_np(xyz):
    """ https://stackoverflow.com/a/4116899 """
    ptsnew = np.hstack((xyz, np.zeros(xyz.shape)))
    xy = xyz[:,0]**2 + xyz[:,1]**2
    ptsnew[:,3] = np.sqrt(xy + xyz[:,2]**2)
    ptsnew[:,4] = np.arctan2(np.sqrt(xy), xyz[:,2]) # for elevation angle defined from Z-axis down
    #ptsnew[:,4] = np.arctan2(xyz[:,2], np.sqrt(xy)) # for elevation angle defined from XY-plane up
    ptsnew[:,5] = np.arctan2(xyz[:,1], xyz[:,0])
    return ptsnew

In [ ]:
def solve_diffusion_FREE():
    """ solve free diffusion """
    function [tensor_data, ADC_1D, particles] = solveDiffusion_free(nP, bfactor, ScanName, nT, D0)

    # parameters
    Size = [nP, 3]

    # initialise
    pos = zeros(Size)
    phi = zeros(Size)

    # run
    scan = Sequence.(ScanName)(bfactor, 'CMR_RBH', nT)
    seq = []
    for dt, Gv in seq:  # (time step, gradient vector)
        phi += gamma*dot(Gv, pos)*dt  # add phase
        pos += randn_limited(Size, 9)*sqrt(2*D0*dt)  # update position

    # compute analytical 1D solution
    ADC_1D = D0  # 1000 terms is sufficient

    # process
    tensor_data = scan.computeResults(phi)
    return 

def solve_diffusion_ONED():
    """ solve oneD diffusion (3D but fix two dimensions) """
    function [tensor_data, ADC_1D, particles] = solveDiffusion_oneD(nP, bfactor, ScanName, nT, D0, spacing)

    # parameters
    spacing_half = 0.5*spacing;

    # initialise
    pos = [zeros(nP, 2), 0.9995*spacing*(rand(nP, 1)-0.5)];
    phi = zeros(nP, 3);
    isValid = true(nP, 1);

    # run
    scan = Sequence.(ScanName)(bfactor, 'CMR_RBH', nT)
    while scan.next()   # step

        # Note: scan.next() performs the step, so we just get the parameters now
        dt = scan.get_dt  # time step
        gG = scan.get_gG  # gamma*gradient

        % phase
        phi = phi + gG.*pos*dt  # allows for scalar or (row-)vector gG

        # position
        pos_new = pos
        step = randn_limited([sum(isValid), 3], 3)*sqrt(2*D0*dt)  # max 3 stdev in each direction
        pos_new(isValid, :) = pos(isValid, :) + step

        # correct
        counter = 0
        needUpdate = isValid  # update only for those that are still valid
        while any(needUpdate)

            # correct
            isAbove = needUpdate & pos_new(:, 3) > +spacing_half
            isBelow = needUpdate & pos_new(:, 3) < -spacing_half
            pos_new(isAbove, 3) = 2*(+spacing_half) - pos_new(isAbove, 3)
            pos_new(isBelow, 3) = 2*(-spacing_half) - pos_new(isBelow, 3)

            # check
            needUpdate = isValid & abs(pos_new(:, 3)) > spacing_half
            counter = counter + 1
            if counter > 20
                isValid = isValid & ~needUpdate  # those who still need update are invalid
                break
        pos = pos_new

    # valid particles
    particles = sum(isValid)
    phi = phi(isValid, :)

    # compute analytical 1D solution
    ADC_1D = scan.compute_analytical(spacing, D0, 1000)  # 1000 terms is sufficient

    # process
    tensor_data = scan.computeResults(phi);


In [1]:
from scipy.stats import truncnorm
def randn_limited(shape, max_std):
    x = truncnorm.rsv(-max_std, +max_std, num)
    return np.reshape(x, shape)


In [2]:
spacing_ra = 1e-6*np.arange(10, 200, 10)
D0 = 3e-9
bfactor = 225e6
[tensor_oneD, ADC_1D, particles] = solveDiffusion_oneD(nP, bfactor, ScanName, nT, D0, spacing)

SyntaxError: invalid syntax (<ipython-input-2-654588c18c1a>, line 1)